<p>
<font size='5' face='Georgia, Arial'>IIC-2233 Apunte Programación Avanzada</font><br>
<font size='1'>&copy; 2015 Karim Pichara - Christian Pieringer. Todos los derechos reservados. Editado por Equipo Docente IIC2233 2018-1 al 2026-2</font>
</p>

# Tabla de contenidos

1. [Ejecución de código de ejemplo del material que usa PyQt](#Ejecución-de-código-de-ejemplo-del-material-que-usa-PyQt)
2. [*Threads* y PyQt](#Threads-y-PyQt)
3. [QThread](#QThread)
    1. [QThread con su propia señal](#QThread-con-su-propia-señal)
4. [Afinidad: ¿en qué *thread* se ejecuta cada método?](#Afinidad:-¿en-qué-thread-se-ejecuta-cada-método?)
    1. [Precaución: ¿dónde se crea un objeto?](#Precaución:-¿dónde-se-crea-un-objeto?)
5. [Múltiples QThread](#Múltiples-QThread)
6. [Detener un QThread y cerrar limpio](#Detener-un-QThread-y-cerrar-limpio)
7. [`QMutex`](#QMutex)

## Ejecución de código de ejemplo del material que usa PyQt

Jupyter no fue creado para ejecutar interfaces gráficas de escritorio, por lo que en este _notebook_ el código se muestra como texto y **no** se ejecuta. Cada ejemplo está en la carpeta `scripts/`, y te recomendamos **fuertemente** ejecutarlo desde tu computador e interactuar con él.

**Importante:** al ejecutar un _script_, la terminal debe estar posicionada en la misma carpeta del archivo `.py`. Por ejemplo:

```bash
cd scripts/3-qthread
python3 1_thread.py
```

## *Threads* y PyQt

Cuando una tarea bloquea o tarda mucho (ver la tabla del primer *notebook*), la ejecutamos en otro *thread*. Lo que aprendimos en la semana de *threading* sigue siendo válido: podemos usar `threading.Thread` junto con PyQt, siempre que respetemos la regla de oro: **el _thread_ no toca la interfaz, sino que emite señales**.

En el siguiente ejemplo, la ventana define una señal y se la entrega al *thread*. El *thread* la emite 10 veces, y la ventana actualiza su *label* en el método conectado, que se ejecuta en el *thread* principal.

Este código se encuentra en el archivo `3-qthread/1_thread.py`.

```python
import sys
from threading import Thread
from time import sleep
from PyQt6.QtCore import pyqtSignal
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QPushButton


class MiThread(Thread):
    """
    Esta clase representa un thread personalizado que será utilizado durante
    la ejecución de la GUI.
    """

    def __init__(self, senal_actualizar: pyqtSignal) -> None:
        super().__init__()
        self.senal_actualizar = senal_actualizar

    def run(self) -> None:
        for i in range(10):
            sleep(0.5)
            self.senal_actualizar.emit(str(i))
        self.senal_actualizar.emit("Status: thread terminado")


class MiVentana(QWidget):
    # Creamos una señal para manejar la respuesta del thread
    senal_thread = pyqtSignal(str)

    def __init__(self) -> None:
        super().__init__()
        self.mi_thread = None
        # Conectamos la señal al método que maneja
        self.senal_thread.connect(self.actualizar_label)
        self.init_gui()

    def init_gui(self) -> None:
        # Configuramos los widgets de la interfaz
        self.label = QLabel("Status: esperando thread", self)
        self.boton = QPushButton("Ejecutar Thread", self)
        self.boton.clicked.connect(self.ejecutar_thread)

        self.label.setGeometry(10, 10, 230, 30)
        self.boton.setGeometry(10, 50, 230, 30)

        # Configuramos las propiedades de la ventana.
        self.setWindowTitle("Ejemplo thread")
        self.setGeometry(50, 50, 250, 200)
        self.show()

    def ejecutar_thread(self) -> None:
        """
        Este método crea un thread cada vez que se presiona el botón en la
        interfaz. El thread recibirá como argumento la señal sobre la cual
        debe operar.
        """
        if self.mi_thread is None or not self.mi_thread.is_alive():
            self.mi_thread = MiThread(self.senal_thread)
            self.mi_thread.start()

    def actualizar_label(self, texto: str) -> None:
        """
        Este método actualiza el label según los datos enviados desde el
        thread a través del argumento texto. Para este ejemplo, el método
        recibe un texto, pero podría también no recibir nada.
        """
        self.label.setText(texto)
```

## QThread

PyQt ofrece su propia clase de *thread*: `QThread`, del módulo `PyQt6.QtCore`. Se usa igual que `Thread`: heredamos de ella, definimos `run` y lo iniciamos con `start()`. La ventaja es que, al ser parte de PyQt, se integra mejor con el resto del *framework*: puede definir sus propias señales, y tiene señales propias como `finished`, que se emite cuando `run` termina.

Las equivalencias principales con `threading.Thread` son:

| `threading.Thread` | `QThread` |
|---|---|
| `start()` | `start()` |
| `run()` | `run()` |
| `join()` | `wait()` |
| `is_alive()` | `isRunning()` |
| `daemon = True` | No existe: hay que detenerlos nosotros (ver [Detener un QThread y cerrar limpio](#Detener-un-QThread-y-cerrar-limpio)) |

El siguiente código es el ejemplo anterior hecho con `QThread`. Este código se encuentra en el archivo `3-qthread/2_qthread.py`.

```python
import sys
from time import sleep
from PyQt6.QtCore import pyqtSignal, QThread
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QPushButton


class MiThread(QThread):
    """
    Esta clase representa un thread personalizado que será utilizado durante
    la ejecución de la GUI.
    """

    def __init__(self, senal_actualizar: pyqtSignal) -> None:
        super().__init__()
        self.senal_actualizar = senal_actualizar

    def run(self) -> None:
        for i in range(10):
            sleep(0.5)
            self.senal_actualizar.emit(str(i))

        sleep(0.5)
        self.senal_actualizar.emit("Status: QThread terminado")
```

El resto del código es igual al ejemplo anterior, excepto que en `ejecutar_thread` usamos `isRunning()` en vez de `is_alive()`. Fíjate también en que guardamos el *thread* en `self.mi_thread`, y no en `self.thread`: todo objeto de PyQt, incluida la ventana, ya tiene un método `thread()` (retorna el *thread* donde vive el objeto), y si lo sobrescribimos con un atributo, ese método deja de funcionar.

### QThread con su propia señal

Como `QThread` es parte de PyQt, puede definir sus propias señales como atributos de clase. Esto **no es posible con `Thread`**, ya que las señales solo pueden definirse en clases que heredan de `QObject`. Así, ya no es necesario que la ventana le entregue su señal al *thread*: basta con conectar la señal del *thread* después de crearlo.

Este código se encuentra en el archivo `3-qthread/3_qthread_con_signal.py`.

```python
import sys
from time import sleep
from PyQt6.QtCore import pyqtSignal, QThread
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QPushButton


class MiThread(QThread):
    """
    Esta clase representa un thread personalizado que será utilizado durante
    la ejecución de la GUI.
    """

    # Creamos una señal para manejar la respuesta del thread
    senal_thread = pyqtSignal(str)

    def __init__(self) -> None:
        super().__init__()

    def run(self) -> None:
        for i in range(10):
            sleep(0.5)
            self.senal_thread.emit(str(i))

        sleep(0.5)
        self.senal_thread.emit("Status: QThread terminado")


class MiVentana(QWidget):

    def __init__(self) -> None:
        super().__init__()
        self.mi_thread = None
        self.init_gui()

    def init_gui(self) -> None:
        # Configuramos los widgets de la interfaz
        self.label = QLabel("Status: esperando QThread", self)
        self.boton = QPushButton("Ejecutar QThread", self)
        self.boton.clicked.connect(self.ejecutar_thread)

        self.label.setGeometry(10, 10, 230, 30)
        self.boton.setGeometry(10, 50, 230, 30)

        # Configuramos las propiedades de la ventana.
        self.setWindowTitle("Ejemplo QThread")
        self.setGeometry(50, 50, 250, 200)
        self.show()

    def ejecutar_thread(self) -> None:
        """
        Este método crea un thread cada vez que se presiona el botón en la
        interfaz, si no hay otro corriendo. Como el thread tiene su propia
        señal, basta con conectarla después de crearlo.
        """
        # Aquí debemos ocupar isRunning en lugar de is_alive
        if self.mi_thread is None or not self.mi_thread.isRunning():
            self.mi_thread = MiThread()
            # Conectamos la señal del thread al método que maneja
            self.mi_thread.senal_thread.connect(self.actualizar_label)
            self.mi_thread.start()

    def actualizar_label(self, texto: str) -> None:
        """
        Este método actualiza el label según los datos enviados desde el
        thread a través del argumento texto. Para este ejemplo, el método
        recibe un texto, pero podría también no recibir nada.
        """
        self.label.setText(texto)
```

## Afinidad: ¿en qué *thread* se ejecuta cada método?

Hay un detalle de `QThread` que causa muchos errores: **un objeto `QThread` no es el _thread_, sino un objeto que administra un _thread_**. Y como todo objeto de PyQt, "vive" en el *thread* donde fue creado. Esto se llama **afinidad**.

Esto tiene dos consecuencias:

1. De un `QThread`, **solo el método `run` se ejecuta en el _thread_ nuevo**. Cualquier otro método del objeto se ejecuta en otro lugar: si lo llamas directamente, en el *thread* que lo llamó (como cualquier función de Python); si está conectado a una señal, en el *thread* donde vive el objeto (normalmente, el principal).
2. Cuando se emite una señal, el método conectado se ejecuta **en el _thread_ donde vive el objeto receptor**, no en el *thread* que emitió la señal. Si son *threads* distintos, la llamada se encola y la ejecuta el *event loop* del *thread* receptor.

Para verlo, el siguiente ejemplo imprime en qué *thread* se ejecuta cada método, usando dos funciones del módulo `threading`: `threading.current_thread()`, que vimos en la semana de *threading* y retorna el *thread* que está ejecutando el código, y `threading.main_thread()`, que no habíamos visto y retorna el *thread* principal del programa (el mismo que ejecuta el *event loop*). Si ambas retornan el mismo objeto, el código se está ejecutando en el *thread* principal. El botón "Pedirle una tarea pesada al QThread" emite una señal conectada al método `tarea_pesada` del `QThread`.

Este código se encuentra en el archivo `3-qthread/4_afinidad.py`.

```python
import sys
import threading
from time import sleep
from PyQt6.QtCore import pyqtSignal, QThread
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QPushButton


def donde_estoy() -> str:
    # Usamos threading (semana 9) para saber en qué thread se ejecuta el código
    if threading.current_thread() is threading.main_thread():
        return "thread PRINCIPAL"
    return "thread SECUNDARIO"


class MiThread(QThread):
    senal_progreso = pyqtSignal(str)

    def run(self) -> None:
        # run es lo único que se ejecuta en el thread nuevo
        print(f"[run] me ejecuto en el {donde_estoy()}")
        for i in range(1, 6):
            sleep(1)
            self.senal_progreso.emit(f"Paso {i} de 5")
        self.senal_progreso.emit("Status: QThread terminado")

    def tarea_pesada(self) -> None:
        # Este método es un slot del QThread. ¿Dónde se ejecuta?
        print(f"[tarea_pesada] me ejecuto en el {donde_estoy()}")
        sleep(3)
        print("[tarea_pesada] terminé")


class MiVentana(QWidget):
    senal_pedir_tarea = pyqtSignal()

    def __init__(self) -> None:
        super().__init__()
        print(f"[__init__ de MiVentana] me ejecuto en el {donde_estoy()}")
        self.mi_thread = MiThread()
        self.mi_thread.senal_progreso.connect(self.actualizar_label)
        self.senal_pedir_tarea.connect(self.mi_thread.tarea_pesada)
        self.init_gui()

    def init_gui(self) -> None:
        # Configuramos los widgets de la interfaz
        self.label = QLabel("Status: esperando QThread", self)
        self.boton_iniciar = QPushButton("Iniciar QThread", self)
        self.boton_tarea = QPushButton("Pedirle una tarea pesada al QThread", self)

        self.label.setGeometry(10, 10, 330, 30)
        self.boton_iniciar.setGeometry(10, 50, 330, 30)
        self.boton_tarea.setGeometry(10, 90, 330, 30)

        self.boton_iniciar.clicked.connect(self.iniciar_thread)
        self.boton_tarea.clicked.connect(self.senal_pedir_tarea.emit)

        # Configuramos las propiedades de la ventana.
        self.setWindowTitle("Afinidad de threads")
        self.setGeometry(50, 50, 350, 130)
        self.show()

    def iniciar_thread(self) -> None:
        if not self.mi_thread.isRunning():
            self.mi_thread.start()

    def actualizar_label(self, texto: str) -> None:
        print(f"[actualizar_label] me ejecuto en el {donde_estoy()}")
        self.label.setText(texto)
```

Si presionas "Iniciar QThread" y luego "Pedirle una tarea pesada al QThread", la consola muestra algo así:

```
[__init__ de MiVentana] me ejecuto en el thread PRINCIPAL
[run] me ejecuto en el thread SECUNDARIO
[actualizar_label] me ejecuto en el thread PRINCIPAL
[tarea_pesada] me ejecuto en el thread PRINCIPAL
[tarea_pesada] terminé
[actualizar_label] me ejecuto en el thread PRINCIPAL
[actualizar_label] me ejecuto en el thread PRINCIPAL
...
```

`tarea_pesada` es un método del `QThread`, pero se ejecutó en el *thread* **principal**, y la ventana se congeló durante 3 segundos. El objeto `MiThread` se creó en el `__init__` de la ventana, es decir, en el *thread* principal, así que vive ahí.

![Afinidad de threads](img/afinidad-threads.svg)

**Conclusión práctica:** el trabajo pesado va **dentro de `run`**. Si necesitas pedirle cosas al *thread* mientras corre, usa atributos que `run` revise en su ciclo (por ejemplo, una bandera o una `Queue`, como vimos en la semana de *threading*), no métodos conectados a señales.

### Precaución: ¿dónde se crea un objeto?

La afinidad también explica un error más sutil. En el siguiente ejemplo, el código principal crea y ejecuta un `QThread` (T1), y **dentro del `run` de T1** se crea y ejecuta otro `QThread` (T2). Luego conectamos `signal_t1` desde el *thread* principal y `signal_t2` desde el *thread* de T1, y tanto T1 como T2 emiten ambas señales. ¿Se ejecutarán las 4?

Este código se encuentra en el archivo `3-qthread/5a_signals_no_funcionan.py`.

```python
import sys
from PyQt6.QtCore import QThread, QTimer, pyqtSignal
from PyQt6.QtWidgets import QApplication


class Thread1(QThread):
    signal_t1 = pyqtSignal(str)

    def __init__(self) -> None:
        super().__init__()
        self.t2 = None

    def run(self) -> None:
        print("[Thread 1] Crear Thread 2")
        # OJO: este objeto se crea DENTRO de run, es decir, en el thread de T1.
        # Por lo tanto, el objeto t2 "vive" (tiene afinidad) en el thread de T1
        self.t2 = Thread2(self.signal_t1)

        print("[Thread 1] Conectar signal_t2 con imprimir de T2")
        self.t2.signal_t2.connect(self.t2.imprimir)

        print("[Thread 1] Ejecutar Thread 2 y esperar a que termine")
        self.t2.start()
        self.t2.wait()

        print("[Thread 1] Emitir signal_t1 y signal_t2")
        self.signal_t1.emit("signal_t1 emitida desde Thread 1")
        self.t2.signal_t2.emit("signal_t2 emitida desde Thread 1")

    def imprimir(self, texto: str) -> None:
        print("\t[imprimir de T1] ->", texto)


class Thread2(QThread):
    signal_t2 = pyqtSignal(str)

    def __init__(self, signal_t1: pyqtSignal) -> None:
        super().__init__()
        self.signal_t1 = signal_t1

    def run(self) -> None:
        print("[Thread 2] Emitir signal_t1 y signal_t2")
        self.signal_t1.emit("signal_t1 emitida desde Thread 2")
        self.signal_t2.emit("signal_t2 emitida desde Thread 2")

    def imprimir(self, texto: str) -> None:
        print("\t[imprimir de T2] ->", texto)


if __name__ == "__main__":

    def hook(type_, value, traceback):
        # Imprimimos el error con el hook original de Python. Al instalar
        # nuestro propio sys.excepthook, PyQt no cierra el programa
        sys.__excepthook__(type_, value, traceback)

    sys.excepthook = hook
    app = QApplication([])
    print("[Main] Crear Thread 1")
    hilo = Thread1()
    print("[Main] Conectar signal_t1 con imprimir de T1")
    hilo.signal_t1.connect(hilo.imprimir)
    print("[Main] Ejecutar Thread 1")
    hilo.start()

    # Cerramos el programa después de 1 segundo
    QTimer.singleShot(1000, app.quit)
    app.exec()
    hilo.wait()
```

Al ejecutarlo se obtiene algo como lo siguiente. El orden de las dos últimas líneas puede variar entre ejecuciones, porque una se ejecuta en el *thread* principal y la otra en el de T1, al mismo tiempo:

```
[Main] Crear Thread 1
[Main] Conectar signal_t1 con imprimir de T1
[Main] Ejecutar Thread 1
[Thread 1] Crear Thread 2
[Thread 1] Conectar signal_t2 con imprimir de T2
[Thread 1] Ejecutar Thread 2 y esperar a que termine
[Thread 2] Emitir signal_t1 y signal_t2
	[imprimir de T1] -> signal_t1 emitida desde Thread 2
[Thread 1] Emitir signal_t1 y signal_t2
	[imprimir de T1] -> signal_t1 emitida desde Thread 1
	[imprimir de T2] -> signal_t2 emitida desde Thread 1
```

Se emitieron 4 señales, pero solo se ejecutaron 3: falta `signal_t2` emitida desde Thread 2. ¿Por qué?

- El objeto `hilo` (T1) se creó en el *thread* principal, así que vive ahí. Cuando T1 o T2 emiten `signal_t1`, la llamada a `hilo.imprimir` se encola en el *thread* principal, que tiene un *event loop* (`app.exec()`) que la ejecuta. ✅
- El objeto `t2` se creó **dentro de `run` de T1**, así que vive en el *thread* de T1.
    - Cuando **T1** emite `signal_t2`, el emisor y el receptor están en el mismo *thread*, así que `t2.imprimir` se ejecuta directamente. ✅
    - Cuando **T2** emite `signal_t2` desde su propio *thread*, la llamada a `t2.imprimir` se encola en el *thread* de T1. Pero el *thread* de T1 **no tiene _event loop_**: está ejecutando nuestro `run`, que nunca llama a `exec()`. Nadie saca ese evento de la cola, así que nunca se ejecuta. ❌

La solución es crear T2 en el *thread* principal, por ejemplo en el `__init__` de T1. Con ese único cambio, que está en `3-qthread/5b_signals_corregido.py`, se ejecutan las 4 señales:

```python
    def __init__(self) -> None:
        super().__init__()
        # CORRECCIÓN: creamos T2 en __init__, que se ejecuta en el thread
        # principal. Así el objeto t2 vive en el thread principal, que sí
        # tiene un event loop (app.exec()) para procesar sus señales
        self.t2 = Thread2(self.signal_t1)

    def run(self) -> None:
        print("[Thread 1] Conectar signal_t2 con imprimir de T2")
        self.t2.signal_t2.connect(self.t2.imprimir)

        print("[Thread 1] Ejecutar Thread 2 y esperar a que termine")
        self.t2.start()
        self.t2.wait()

        print("[Thread 1] Emitir signal_t1 y signal_t2")
        self.signal_t1.emit("signal_t1 emitida desde Thread 1")
        self.t2.signal_t2.emit("signal_t2 emitida desde Thread 1")
```

**Regla práctica:** crea los objetos de PyQt (`QThread`, `QTimer`, `QObject` con señales, *widgets*) en el *thread* principal, por ejemplo en un `__init__` o en un *slot*, y **no dentro de `run`**. Por la misma razón, un `QTimer` creado dentro de `run` tampoco funciona: su `timeout` necesita un *event loop* en ese *thread*.

## Múltiples QThread

Ahora queremos tener una cantidad variable de `QThread` corriendo a la vez. Lo más importante es **siempre guardar una referencia** a cada `QThread` mientras corre, por ejemplo en una lista. Este es el mismo ejemplo que hicimos con varios `QTimer` en el *notebook* anterior.

Este código se encuentra en el archivo `3-qthread/6_multiples_qthreads.py`.

```python
import sys
from time import sleep
from PyQt6.QtCore import pyqtSignal, QThread
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QPushButton


class MiThread(QThread):
    # Se define para la clase MiThread,
    # para que cada instancia tenga una propia
    senal_actualizar = pyqtSignal(int, str)

    def __init__(self, indice: int, tiempo: float) -> None:
        super().__init__()
        self.indice = indice
        self.tiempo = tiempo

    def run(self) -> None:
        for i in range(10):
            sleep(self.tiempo)
            self.senal_actualizar.emit(self.indice, str(i))

        sleep(self.tiempo)
        self.senal_actualizar.emit(self.indice, "Status: QThread terminado")


class MiVentana(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.threads = []
        self.init_gui()

    def init_gui(self) -> None:
        # Configuramos los widgets de la interfaz
        # Definimos un montón de labels que corresponderán a un thread cada uno
        self.labels = {
            i: QLabel("Status: esperando QThread", self) for i in range(1, 6)
        }
        self.boton = QPushButton("Ejecutar QThreads", self)
        self.boton.clicked.connect(self.ejecutar_threads)

        for i in range(1, 6):
            self.labels[i].setGeometry(10, (i - 1) * 30, 330, 30)

        self.boton.setGeometry(10, 150, 330, 30)
        # Configuramos las propiedades de la ventana.
        self.setWindowTitle("Ejemplo Multiples QThreads")
        self.setGeometry(50, 50, 350, 200)
        self.show()

    def ejecutar_threads(self) -> None:
        """
        Este método crea cinco threads cada vez que se presiona el botón en la
        interfaz. Los threads recibirán como argumento el índice del label
        que les corresponde y el tiempo que toman entre cada iteración.
        """
        for thread in self.threads:
            if thread.isRunning():
                return

        self.threads = []
        for i in range(1, 6):
            thread = MiThread(i, i / 10)
            # Se conecta la señal emitida por el thread a un método
            # de la ventana
            thread.senal_actualizar.connect(self.actualizar_labels)
            self.threads.append(thread)
            thread.start()

    def actualizar_labels(self, indice: int, texto: str) -> None:
        """
        Este método actualiza el label correspondiente según los datos
        enviados desde un thread a través del índice y aplica el texto.
        """
        self.labels[indice].setText(texto)
```

¿Qué pasa si no guardamos la referencia? Por ejemplo:

```python
def ejecutar_thread(self) -> None:
    thread = MiThread()  # ¡Solo una variable local!
    thread.senal_thread.connect(self.actualizar_label)
    thread.start()
```

Al terminar el método, la variable local desaparece y Python puede eliminar el objeto `QThread` mientras su *thread* sigue corriendo. El programa se cae con:

```
QThread: Destroyed while thread '' is still running
Aborted (core dumped)
```

Lo mismo puede ocurrir si reasignas el atributo (`self.mi_thread = MiThread()`) mientras el *thread* anterior sigue corriendo. Por eso los ejemplos revisan `isRunning()` antes de crear uno nuevo.

## Detener un QThread y cerrar limpio

A diferencia de `threading.Thread`, un `QThread` no puede ser *daemon*. Si el *thread* tiene un ciclo infinito, debemos detenerlo nosotros. ¿Por qué importa?

- **Al volver a un menú.** Es muy común que en una tarea la ventana del juego se cierre para volver a la ventana de inicio. Si los *threads* del juego no se detienen, siguen corriendo en el fondo, y al empezar una partida nueva tendrás dos juegos corriendo a la vez.
- **Al cerrar el programa.** El *thread* se corta abruptamente en cualquier punto de su ejecución, por ejemplo a la mitad de escribir un archivo. Con un `threading.Thread` que no es *daemon* pasa lo contrario: el programa no termina hasta que el *thread* termine. Puedes verlo en `3-qthread/1_thread.py`: si cierras la ventana mientras el *thread* cuenta, la terminal queda esperando unos segundos antes de terminar.

La forma recomendada es usar una **bandera** que el ciclo de `run` revise en cada iteración, y luego esperar a que el *thread* termine con `wait()` (el equivalente a `join()`). En el siguiente ejemplo lo hacemos en `closeEvent`, el método que PyQt llama cuando el usuario cierra una ventana.

Este código se encuentra en el archivo `3-qthread/7_cierre_limpio.py`.

```python
import sys
from time import sleep
from PyQt6.QtCore import pyqtSignal, QThread
from PyQt6.QtGui import QCloseEvent
from PyQt6.QtWidgets import QApplication, QWidget, QLabel


class Contador(QThread):
    senal_actualizar = pyqtSignal(str)

    def __init__(self) -> None:
        super().__init__()
        # Bandera que el thread revisa en cada iteración de su ciclo
        self.activo = True

    def run(self) -> None:
        segundos = 0
        while self.activo:
            # Dormimos de a poco (0.1 s) para revisar la bandera seguido.
            # Si durmiéramos 10 segundos, el cierre tardaría hasta 10 segundos
            sleep(0.1)
            segundos += 0.1
            self.senal_actualizar.emit(f"Llevo {segundos:.1f} segundos")
        print("[Contador] salí del ciclo: run terminó")

    def detener(self) -> None:
        self.activo = False


class MiVentana(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.label = QLabel("Status: iniciando", self)
        self.label.setGeometry(10, 10, 230, 30)
        self.setWindowTitle("Cierre limpio de un QThread")
        self.setGeometry(50, 50, 250, 50)

        self.contador = Contador()
        self.contador.senal_actualizar.connect(self.label.setText)
        self.contador.start()
        self.show()

    def closeEvent(self, evento: QCloseEvent) -> None:
        """
        closeEvent se llama cuando el usuario cierra la ventana. Antes de
        cerrar, le pedimos al thread que termine y lo esperamos con wait(),
        el equivalente a join() de threading.
        """
        print("[Ventana] cerrando: le pido al thread que se detenga")
        self.contador.detener()
        self.contador.wait()
        print("[Ventana] el thread terminó, ahora sí cierro")
        evento.accept()
```

Al cerrar la ventana, la consola muestra:

```
[Ventana] cerrando: le pido al thread que se detenga
[Contador] salí del ciclo: run terminó
[Ventana] el thread terminó, ahora sí cierro
```

Algunos detalles importantes:

- El `sleep` dentro del ciclo es corto (0,1 s), para que el *thread* revise la bandera seguido. Si durmiera 10 segundos, `wait()` podría bloquear la ventana hasta 10 segundos.
- `QThread` tiene un mecanismo equivalente a nuestra bandera: `requestInterruption()` para pedir que se detenga e `isInterruptionRequested()` para revisarlo dentro de `run`.
- Existe un método `terminate()` que mata el *thread* de inmediato, pero **no lo uses**: puede dejar *locks* tomados y datos a medio escribir. La propia documentación de Qt lo desaconseja.

## `QMutex`

Así como existe `QThread`, PyQt tiene su propia versión del `Lock`: `QMutex`, con los métodos `lock()` y `unlock()`. Permite que solo un *thread* a la vez ejecute una sección crítica del código. Igual que con `acquire()` y `release()` del `Lock`, conviene llamar a `unlock()` dentro de un `finally`: si ocurre una excepción en la sección crítica y el *mutex* no se libera, los demás *threads* quedan esperando para siempre.

En el siguiente ejemplo creamos 5 *threads* que actualizan la ventana a distintas velocidades. El primer `for` de `run` está protegido por un `QMutex` compartido, por lo que solo un *thread* a la vez puede ejecutarlo. El segundo `for` no está protegido. Al ejecutarlo verás que, mientras varios *threads* avanzan en su parte "no crítica", solo uno a la vez avanza en su parte "crítica".

Este código se encuentra en el archivo `3-qthread/8_qmutex.py`.

```python
import sys
from time import sleep
from PyQt6.QtCore import pyqtSignal, QThread, QMutex
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QPushButton


class MiThread(QThread):
    # Se define para la clase MiThread,
    # para que cada instancia tenga una propia
    senal_actualizar = pyqtSignal(int, str)

    def __init__(self, indice: int, tiempo: float, mutex: QMutex) -> None:
        super().__init__()
        self.indice = indice
        self.tiempo = tiempo
        self.mutex = mutex

    def run(self) -> None:
        # Sección crítica: solo un thread a la vez. Liberamos el mutex en un
        # finally, para que se libere aunque ocurra una excepción. Si no, los
        # demás threads quedarían esperando para siempre
        self.mutex.lock()
        try:
            for i in range(10):
                sleep(self.tiempo)
                self.senal_actualizar.emit(self.indice, str(i) + " - crítico")
        finally:
            self.mutex.unlock()

        for i in range(50):
            sleep(self.tiempo)
            self.senal_actualizar.emit(self.indice, str(i) + " - no crítico")

        sleep(self.tiempo)
        self.senal_actualizar.emit(self.indice, "Status: QThread terminado")
```

Dos comentarios:

- El `Lock` de `threading` (incluyendo `with lock:`) funciona perfectamente con `QThread`. Puedes usar el que prefieras, pero siempre debe ser **el mismo objeto** compartido por todos los *threads*.
- Si respetas la regla de oro y los *threads* solo se comunican mediante señales, la mayor parte del estado del programa se modifica en el *thread* principal, y rara vez vas a necesitar un *lock*.